In [ ]:
import os
import time
import requests
import pandas as pd
from datetime import timedelta
import finnhub
from dotenv import load_dotenv

load_dotenv()

In [ ]:
FINN_KEY = os.environ["FINNHUB_API_KEY"]
DATA_DIR = "../data/raw"
BASE_CSV = os.path.join(DATA_DIR, "earnings_base.csv")
OUTPUT_CSV = "raw_news_for_colab.csv"

In [ ]:
DAYS_BEFORE = 14
DAYS_AFTER = 0
MAX_ARTICLES_PER_WINDOW = 10

COMPANY_NAMES = {
    'AAPL':  'Apple',
    'MSFT':  'Microsoft',
    'GOOGL': 'Google Alphabet',
    'AMZN':  'Amazon',
    'META':  'Meta Facebook',
    'NVDA':  'Nvidia',
    'AVGO':  'Broadcom',
    'MU':    'Micron Technology',
    'AMD':   'AMD Advanced Micro Devices',
    'AMAT':  'Applied Materials',
}

In [ ]:
df_earnings = pd.read_csv(BASE_CSV)
df_earnings['earnings_date'] = pd.to_datetime(df_earnings['earnings_date'])
print(f'Loaded {len(df_earnings)} earnings rows')
print(f'Date range: {df_earnings["earnings_date"].min().date()} → {df_earnings["earnings_date"].max().date()}')
print(f'Tickers: {sorted(df_earnings["ticker"].unique().tolist())}')
df_earnings.head()

In [ ]:
finnhub_client = finnhub.Client(api_key=FINN_KEY)

extracted_text_rows = []

for idx, row in df_earnings.iterrows():
    ticker = row['ticker']
    earnings_date = row['earnings_date']
    
    date_to = earnings_date - timedelta(days=0)
    date_from = earnings_date - timedelta(days=DAYS_BEFORE)

    try:
        articles = finnhub_client.company_news(
            ticker,
            _from=date_from.strftime('%Y-%m-%d'),
            to=date_to.strftime('%Y-%m-%d')
        )
        
        for article in articles[:MAX_ARTICLES_PER_WINDOW]:
            title = article.get('headline', '')
            description = article.get('summary', '')
            
            extracted_text_rows.append({
                'ticker': ticker,
                'earnings_date': earnings_date.date(),
                'full_text': f"{title}. {description}"
            })

    except Exception as e:
        print(f"Error downloading text for {ticker}: {str(e)}")
        continue

    time.sleep(1.2)

df_colab_ready = pd.DataFrame(extracted_text_rows)
df_colab_ready.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Extraction Complete! Upload '{OUTPUT_CSV}' ({len(df_colab_ready)} text rows) to Google Colab.")